In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import BaggingRegressor

# 1. Read the data
data = pd.read_csv('spotify_data.csv')

# 2. Drop the specified variables
data = data.drop(columns=['id', 'performer', 'song', 'genre', 'key'])

# 3. Separate outcome (y) and features (X)
y = data['rating']
X = data.drop(columns=['rating'])

# 4. Create binned outcome variable (20 equally sized bins)
y_binned = pd.qcut(data['rating'], q=20)

# 5. Stratified random sampling (70% train, 30% test, seed=1031)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.30, 
    random_state=1031, 
    stratify=y_binned
)

# 6. Calculate the median value of liveness in the train sample
median_liveness_train = X_train['liveness'].median()
print(median_liveness_train)

0.131


In [2]:


# 2. 仅提取十个 genre 相关的哑变量作为自变量
genre_features = [
    'genre_country', 'genre_blues', 'genre_jazz', 'genre_rap', 
    'genre_space', 'genre_dance', 'genre_pop', 'genre_rock', 
    'genre_hip_hop', 'genre_rnb'
]
X_train_genre = X_train[genre_features]

# 3. 建立并拟合线性回归模型
model = LinearRegression()
model.fit(X_train_genre, y_train)

# 4. 在训练集上进行预测
y_pred_train = model.predict(X_train_genre)

# 5. 计算并输出均方根误差 (RMSE)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
print(rmse_train)

15.811872341573341


In [3]:

# 3. 建立并拟合最大深度为 4 的决策树回归模型
# (注意：凡是涉及随机性的算法都需要设置 random_state=1031)
tree_model = DecisionTreeRegressor(max_depth=4, random_state=1031)
tree_model.fit(X_train_genre, y_train)

# 4. 在训练集上进行预测
y_pred_train_tree = tree_model.predict(X_train_genre)

# 5. 计算并输出均方根误差 (RMSE)
rmse_train_tree = np.sqrt(mean_squared_error(y_train, y_pred_train_tree))
print(rmse_train_tree)

15.691917118321479


In [4]:

# 3. 实例化两个基模型 (Base estimators)
# (确保决策树同样使用 random_state=1031)
lr = LinearRegression()
dt = DecisionTreeRegressor(max_depth=4, random_state=1031)

# 4. 构建并拟合 Voting Regressor
voting_model = VotingRegressor(estimators=[('lr', lr), ('dt', dt)])
voting_model.fit(X_train_genre, y_train)

# 5. 在训练集上进行预测
y_pred_train_voting = voting_model.predict(X_train_genre)

# 6. 计算并输出均方根误差 (RMSE)
rmse_train_voting = np.sqrt(mean_squared_error(y_train, y_pred_train_voting))
print(rmse_train_voting)

15.67081869948398


In [5]:
# 2. 定义基估计器：回归树 (max_depth=10)
dt = DecisionTreeRegressor(max_depth=10, random_state=1031)

# 3. 定义并拟合 Bagging 模型 (n_estimators=100)
bagging_model = BaggingRegressor(
    estimator=dt,  # 在较老版本的 sklearn 中可能需要写成 base_estimator=dt
    n_estimators=100, 
    random_state=1031
)
bagging_model.fit(X_train, y_train)

# 4. 在训练集上进行预测
y_pred_train_bagging = bagging_model.predict(X_train)

# 5. 计算并输出均方根误差 (RMSE)
rmse_train_bagging = np.sqrt(mean_squared_error(y_train, y_pred_train_bagging))
print(rmse_train_bagging)

12.865571490101619


In [6]:
# 2. 定义基估计器：线性回归
lr = LinearRegression()

# 3. 定义并拟合 Bagging 模型 (n_estimators=100)
bagging_lr = BaggingRegressor(
    estimator=lr,  # 或 base_estimator=lr，取决于 sklearn 版本
    n_estimators=100, 
    random_state=1031
)
bagging_lr.fit(X_train, y_train)

# 4. 在训练集上进行预测
y_pred_train_blr = bagging_lr.predict(X_train)

# 5. 计算并输出均方根误差 (RMSE)
rmse_train_blr = np.sqrt(mean_squared_error(y_train, y_pred_train_blr))
print(rmse_train_blr)

15.349484075963746


In [7]:
from sklearn.ensemble import RandomForestRegressor

# 2. 建立随机森林回归模型
# n_estimators=100 表示构建 100 棵决策树 (100 bootstrapped samples)
rf = RandomForestRegressor(n_estimators=100, random_state=1031)
rf.fit(X_train, y_train)

# 3. 在训练集上进行预测
y_pred_train_rf = rf.predict(X_train)

# 4. 计算并输出均方根误差 (RMSE)
rmse_train_rf = np.sqrt(mean_squared_error(y_train, y_pred_train_rf))
print(rmse_train_rf)

5.6700872064569445


In [8]:
# 3. 提取特征重要性 (Feature Importances)
importances = rf.feature_importances_
feature_names = X.columns

# 4. 创建 DataFrame 并按重要性降序排列
feature_importance_df = pd.DataFrame({
    'Feature': feature_names, 
    'Importance': importances
})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# 打印排名前列的特征
print(feature_importance_df.head(10))

             Feature  Importance
0     track_duration    0.149303
4           loudness    0.098916
2       danceability    0.087146
7       acousticness    0.084357
11             tempo    0.083714
9           liveness    0.082993
10           valence    0.080345
3             energy    0.080128
6        speechiness    0.079712
8   instrumentalness    0.052338


In [9]:
# 2. Bagging with Linear Regression
lr = LinearRegression()
bag_lr = BaggingRegressor(estimator=lr, n_estimators=100, random_state=1031, oob_score=True)
bag_lr.fit(X_train, y_train)

# 3. Random Forest 
rf = RandomForestRegressor(n_estimators=100, random_state=1031, oob_score=True)
rf.fit(X_train, y_train)

# 4. Bagging with Regression Tree (max_depth=10)
dt = DecisionTreeRegressor(max_depth=10, random_state=1031)
bag_dt = BaggingRegressor(estimator=dt, n_estimators=100, random_state=1031, oob_score=True)
bag_dt.fit(X_train, y_train)

# 输出 OOB Scores
print(f"Bag with Linear Regression OOB Score: {bag_lr.oob_score_}")
print(f"Random Forest OOB Score: {rf.oob_score_}")
print(f"Bag with Regression Tree OOB Score: {bag_dt.oob_score_}")

Bag with Linear Regression OOB Score: 0.14332921730652493
Random Forest OOB Score: 0.15374993085099264
Bag with Regression Tree OOB Score: 0.17204464483528759


In [11]:
from sklearn.ensemble import AdaBoostRegressor

# 2. 建立 AdaBoost 回归模型
# n_estimators=100 表示使用 100 个决策树桩/树
boost_model = AdaBoostRegressor(n_estimators=100, random_state=1031)
boost_model.fit(X_train, y_train)

# 3. 在训练集上进行预测
y_pred_train = boost_model.predict(X_train)

# 4. 计算并输出均方根误差 (RMSE)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
print(rmse_train)

15.358993566704216


In [12]:
from sklearn.ensemble import GradientBoostingRegressor

# 2. 建立 Gradient Boosting 回归模型
# n_estimators=100 表示使用 100 棵回归树
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=1031)
gb_model.fit(X_train, y_train)

# 3. 在训练集上进行预测
y_pred_train_gb = gb_model.predict(X_train)

# 4. 计算并输出均方根误差 (RMSE)
rmse_train_gb = np.sqrt(mean_squared_error(y_train, y_pred_train_gb))
print(rmse_train_gb)

14.74682887905972


In [15]:
df = pd.read_csv('spotify_data.csv')

df_model = df.drop(columns=['id', 'performer', 'song', 'genre'])

# 将布尔变量转为0/1
for col in df_model.select_dtypes(include='bool').columns:
    df_model[col] = df_model[col].astype(int)

# 定义自变量和因变量
X = df_model.drop(columns=['rating'])
y = df_model['rating']

# 构建模型（Stochastic Gradient Boosting）
model = GradientBoostingRegressor(
    n_estimators=100,
    max_features=0.4,
    subsample=0.6,
    random_state=1031
)

# 训练模型
model.fit(X, y)

# 预测（train sample）
y_pred = model.predict(X)

# 计算 RMSE（不要手动round）
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(rmse)

14.857632573466653
